# OS vulnerabilities

**What is exploitable, where does risk concentrate, and what moved since the last scan?**

The analyst's page. Deliberately not a findings table — Wiz already has one of those, and
the result grid at the bottom of `02_program_performance` is a better one than this
notebook would write. What is here is the shape of the register.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
PAGE = {
    "group_by": ("subscription_name", panels.GROUP_DIMENSIONS),
    "top_n": ("5", ["3", "5", "8", "12"]),
}

import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone(**panels.last_scan(spark, ctx).first().asDict()))

In [ ]:
displayHTML(tiles.register_kpis(panels.register_totals(spark, ctx).first()))

## Severity breakdown

Counts as of the pinned scan, with the move since the previous one. Severity is carried
by a dot **and** the word, here and everywhere: the palette is a heat ramp, and HIGH and
MEDIUM sit 1.6 ΔE apart under deuteranopia. That is measured, not assumed.

In [ ]:
displayHTML(
    tiles.stat_cards([r.asDict() for r in panels.severity_cards(spark, ctx).collect()])
)

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.severity_trend(spark, ctx).toPandas(),
            "scan_ts",
            figures.severity_series(
                {s: s for s in ctx.severities},
                __import__('config').SEVERITY_COLORS,
            ),
        ),
        "Open findings per severity, per scan. Each severity carries its own marker shape, so the lines stay distinguishable without relying on the heat ramp.",
    )
)

## Exploitability & priority

Over **open** lifecycles only. One finding can carry several signals, so these do not
partition anything and they do not sum to the any-of total.

Each tile carries the count that was *never captured* beside it, because a missing signal
is not a negative one. The fourth tile is honest rather than absent: brick does not ingest
internet exposure at all.

In [ ]:
displayHTML(tiles.exploit_kpis(panels.exploit_tiles(spark, ctx).first()))

Chart ▸ Bar (stacked) · X=bucket · Y=findings · Group by=severity · Order=bucket_rank ·
Legend=bottom · X title=Age of open findings · Y title=Lifecycles

## Aging of open findings

How long the open backlog has been open, measured from first detection as of the pinned
scan. Buckets are inclusive on the left: exactly 7.0 days is `0-7d`, 7.01 is `8-30d`.

In [ ]:
display(panels.open_age_buckets(spark, ctx))

## Scan-over-scan movement

In [ ]:
displayHTML(tiles.movement_minis(panels.movement(spark, ctx).first()))

## Where risk concentrates

The top groups keep one colour across both charts below. Five, and one pooled *Other* —
the tail is folded in rather than dropped, so the slices still sum to the register, and
five is as many hues as survive a colourblind check.

The share chart is a pie rather than a doughnut: the total already lives in the KPI band
at the top of this notebook, so a hole would have nothing to hold.

In [ ]:
figures.render(
    figures.describe(
        figures.pie(
            panels.group_mix(
                spark, ctx, ctx.param('group_by'), top_n=ctx.int_param('top_n', 5)
            ).toPandas().rename(
                columns={"group_value": "label", "open": "value"}
            ),
        ),
        "Share of open lifecycles by group, largest first, with the tail pooled into Other.",
    )
)

The trend below counts **findings the API returned**, not lifecycles: the ledger keeps no
per-scan history by group. A group's series drops when its findings stop being returned,
which is usually — but not always — the day they were remediated.

In [ ]:
_order = panels.group_palette(spark, ctx, ctx.param('group_by'),
                             top_n=ctx.int_param('top_n', 5))
figures.render(
    figures.describe(
        figures.trend(
            panels.group_trend(
                spark, ctx, ctx.param('group_by'), top_n=ctx.int_param('top_n', 5)
            ).toPandas().pivot(index="scan_ts", columns="group_value",
                               values="open").reset_index(),
            "scan_ts",
            figures.group_series(_order),
        ),
        "Open findings per group, per scan, from each scan's API snapshot. Each group keeps the hue and marker it has in the share chart above.",
    )
)

Chart ▸ Pivot Table · Rows=group_value · Columns=severity · Value=open · Order=sev_rank

GAS has an expandable tree here. A notebook output cannot expand, so this is flattened to
one level and says so — the pivot below is the honest equivalent, and the picker lets you
swap the axes.

In [ ]:
display(panels.group_severity(spark, ctx, ctx.param('group_by')))